In [1]:
from pathlib import Path
import pandas as pd
import json

In [2]:
PROJECT_ROOT = Path(
    r"D:\dev\projects\fourlang_translation"
)


RAW_OPUS_PATH = (
    PROJECT_ROOT
    /
    "data"
    /
    "raw"
    /
    "en_uz"
    /
    "opus"
    /
    "opus100_en_uz.csv"
)


OUTPUT_DIR = (
    PROJECT_ROOT
    /
    "data"
    /
    "clean"
    /
    "en_uz"
    /
    "exp1"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print(RAW_OPUS_PATH)

print(OUTPUT_DIR)

D:\dev\projects\fourlang_translation\data\raw\en_uz\opus\opus100_en_uz.csv
D:\dev\projects\fourlang_translation\data\clean\en_uz\exp1


In [3]:
opus_df = pd.read_csv(
    RAW_OPUS_PATH
)


print(opus_df.shape)

opus_df.head()

(173157, 2)


,src_en,tgt_uz
0,Printer '%s' is out of paper.,'%s' printerda qogʻoz tugadi.
1,But surely he who bears patiently and is forgi...,Яхшилар сифати бўлган ушбу сифатларга такрор-т...
2,Intersect Paths,Obʼektlarni guruhlash
3,Lodge them where you lodge according to your m...,"Агар ҳомиладор бўлсалар, то ҳомилаларини қўйгу..."
4,"If He willed, He could still the wind, and the...","Агар У зот хоҳласа, шамолни тўхтатиб қўюр. Бас..."


In [4]:
opus_df.columns

Index(['src_en', 'tgt_uz'], dtype='str')

In [5]:
opus_df = opus_df.dropna()


opus_df["src_en"] = (
    opus_df["src_en"]
    .astype(str)
    .str.strip()
)


opus_df["tgt_uz"] = (
    opus_df["tgt_uz"]
    .astype(str)
    .str.strip()
)


# 删除空字符串

opus_df = opus_df[
    (opus_df["src_en"]!="")
    &
    (opus_df["tgt_uz"]!="")
]


# 删除重复

opus_df = opus_df.drop_duplicates()


print(opus_df.shape)

(148655, 2)


In [7]:
pairs = opus_df.sample(
    n=10000,
    random_state=42
)


pairs.head()

,src_en,tgt_uz
50609,New Hard Disc,Yangi qattiq disk uskunasiName
144529,Everyone shall taste the death.,Ҳар бир жон ўлимни топажакдир.
171784,These people say.,"Албатта, анавилар дерлар:"
79241,The present life is nothing but sport and amus...,Бу дунё ҳаёти фақат ўйин-кулгидан иборатдир.
80386,And what will show you what is Saqar?,Ва сақар нималигини сенга нима билдирди?


In [8]:
en_uz_df = pd.DataFrame(
    {
        "src_lang":
        "en",

        "tgt_lang":
        "uz",

        "src_text":
        pairs["src_en"],

        "tgt_text":
        pairs["tgt_uz"]
    }
)

In [9]:
en_uz_df = pd.DataFrame(
    {
        "src_lang":
        "en",

        "tgt_lang":
        "uz",

        "src_text":
        pairs["src_en"],

        "tgt_text":
        pairs["tgt_uz"]
    }
)

In [10]:
uz_en_df = pd.DataFrame(
    {
        "src_lang":
        "uz",

        "tgt_lang":
        "en",

        "src_text":
        pairs["tgt_uz"],

        "tgt_text":
        pairs["src_en"]
    }
)

In [11]:
exp1_df = pd.concat(
    [
        en_uz_df,
        uz_en_df
    ],
    ignore_index=True
)


print(exp1_df.shape)


exp1_df.head()

(20000, 4)


,src_lang,tgt_lang,src_text,tgt_text
0,en,uz,New Hard Disc,Yangi qattiq disk uskunasiName
1,en,uz,Everyone shall taste the death.,Ҳар бир жон ўлимни топажакдир.
2,en,uz,These people say.,"Албатта, анавилар дерлар:"
3,en,uz,The present life is nothing but sport and amus...,Бу дунё ҳаёти фақат ўйин-кулгидан иборатдир.
4,en,uz,And what will show you what is Saqar?,Ва сақар нималигини сенга нима билдирди?


In [12]:
exp1_df = exp1_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [13]:
train_size = int(
    len(exp1_df)*0.8
)


valid_size = int(
    len(exp1_df)*0.1
)



train_df = exp1_df[
    :train_size
]


valid_df = exp1_df[
    train_size:
    train_size+valid_size
]


test_df = exp1_df[
    train_size+valid_size:
]


print(
    train_df.shape,
    valid_df.shape,
    test_df.shape
)

(16000, 4) (2000, 4) (2000, 4)


In [14]:
train_df.to_json(
    OUTPUT_DIR / "train.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)


valid_df.to_json(
    OUTPUT_DIR / "validation.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)


test_df.to_json(
    OUTPUT_DIR / "test.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

In [15]:
with open(
    OUTPUT_DIR/"train.jsonl",
    "r",
    encoding="utf-8"
) as f:

    for i in range(3):
        print(
            json.loads(
                f.readline()
            )
        )

{'src_lang': 'uz', 'tgt_lang': 'en', 'src_text': 'Ҳеч шубҳа йўқки, уларга, албатта, дўзах бўлур ва, албатта, улар (дўзах) пешқадамларидир.', 'tgt_text': '(Tafsir Al-Qurtubi, Vol. 10, Page 121)'}
{'src_lang': 'en', 'tgt_lang': 'uz', 'src_text': 'Remember _forever', 'tgt_text': '_Doim eslab qolish'}
{'src_lang': 'en', 'tgt_lang': 'uz', 'src_text': 'So do not speak too softly, lest the sick at heart lusts after you, but speak in an appropriate manner.', 'tgt_text': 'Овозингизни майин, назокатли қилиб, эркак кишига таъсир этадиган ҳолатда гапирманг. Мунофиқ ва иймони заифлар сизнинг майин овозингизни эшитиб, фисқу фужурни бошлаб қолмасинлар.)'}


In [16]:
opus_df.columns

Index(['src_en', 'tgt_uz'], dtype='str')